In [365]:
# 客户端初始化
from dotenv import load_dotenv
import voyageai

load_dotenv()

client = voyageai.Client()

In [366]:
# 按章节分块
import re


def chunk_by_section(document_text):
    pattern = r"\n## "
    return re.split(pattern, document_text)

In [367]:
# 嵌入向量生成
def generate_embedding(chunks, model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    input = chunks if is_list else [chunks]
    result = client.embed(input, model=model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

In [368]:
# VectorIndex 向量索引实现
import math
from typing import Optional, Any, List, Dict, Tuple


class VectorIndex:
    """基于向量的文档索引，支持余弦距离或欧氏距离的相似度检索。"""

    def __init__(
        self,
        distance_metric: str = "cosine",
        embedding_fn=None,
    ):
        """
        初始化向量索引。
        :param distance_metric: 距离度量，'cosine'（余弦距离）或 'euclidean'（欧氏距离）
        :param embedding_fn: 可选，将文本转为向量的函数；用于 add_document 与 search(query 为字符串时)
        """
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._vector_dim: Optional[int] = None
        if distance_metric not in ["cosine", "euclidean"]:
            raise ValueError("distance_metric 只能是 'cosine' 或 'euclidean'")
        self._distance_metric = distance_metric
        self._embedding_fn = embedding_fn

    def add_document(self, document: Dict[str, Any]):
        """添加一篇文档：用 embedding_fn 对 document['content'] 生成向量后存入索引。"""
        if not self._embedding_fn:
            raise ValueError("初始化时未提供嵌入函数。")
        if not isinstance(document, dict):
            raise TypeError("document 必须是字典。")
        if "content" not in document:
            raise ValueError("文档字典必须包含 'content' 键。")

        content = document["content"]
        if not isinstance(content, str):
            raise TypeError("文档的 'content' 必须是字符串。")

        vector = self._embedding_fn(content)
        self.add_vector(vector=vector, document=document)

    def search(
        self, query: Any, k: int = 1
    ) -> List[Tuple[Dict[str, Any], float]]:
        """
        检索与 query 最相近的 k 个文档。
        :param query: 查询，可为字符串（会经 embedding_fn 转成向量）或向量列表
        :param k: 返回的最相关文档数量
        :return: [(文档字典, 距离), ...]，按距离升序
        """
        if not self.vectors:
            return []

        if isinstance(query, str):
            if not self._embedding_fn:
                raise ValueError("对字符串查询未提供嵌入函数。")
            query_vector = self._embedding_fn(query)
        elif isinstance(query, list) and all(
            isinstance(x, (int, float)) for x in query
        ):
            query_vector = query
        else:
            raise TypeError("query 必须是字符串或数字列表。")

        if self._vector_dim is None:
            return []

        if len(query_vector) != self._vector_dim:
            raise ValueError(
                f"查询向量维度不匹配：期望 {self._vector_dim}，实际 {len(query_vector)}"
            )

        if k <= 0:
            raise ValueError("k 必须是正整数。")

        if self._distance_metric == "cosine":
            dist_func = self._cosine_distance
        else:
            dist_func = self._euclidean_distance

        distances = []
        for i, stored_vector in enumerate(self.vectors):
            distance = dist_func(query_vector, stored_vector)
            distances.append((distance, self.documents[i]))

        distances.sort(key=lambda item: item[0])

        return [(doc, dist) for dist, doc in distances[:k]]

    def add_vector(self, vector, document: Dict[str, Any]):
        """直接添加一个 (向量, 文档) 对，不调用嵌入函数。文档需含 'content' 键。"""
        if not isinstance(vector, list) or not all(
            isinstance(x, (int, float)) for x in vector
        ):
            raise TypeError("vector 必须是数字列表。")
        if not isinstance(document, dict):
            raise TypeError("document 必须是字典。")
        if "content" not in document:
            raise ValueError("文档字典必须包含 'content' 键。")

        if not self.vectors:
            self._vector_dim = len(vector)
        elif len(vector) != self._vector_dim:
            raise ValueError(
                f"向量维度不一致：期望 {self._vector_dim}，实际 {len(vector)}"
            )

        self.vectors.append(list(vector))
        self.documents.append(document)

    def _euclidean_distance(
        self, vec1: List[float], vec2: List[float]
    ) -> float:
        """计算两向量的欧氏距离（L2）。"""
        if len(vec1) != len(vec2):
            raise ValueError("两个向量维度必须相同")
        return math.sqrt(sum((p - q) ** 2 for p, q in zip(vec1, vec2)))

    def _dot_product(self, vec1: List[float], vec2: List[float]) -> float:
        """计算两向量的内积。"""
        if len(vec1) != len(vec2):
            raise ValueError("两个向量维度必须相同")
        return sum(p * q for p, q in zip(vec1, vec2))

    def _magnitude(self, vec: List[float]) -> float:
        """计算向量的 L2 范数（模长）。"""
        return math.sqrt(sum(x * x for x in vec))

    def _cosine_distance(self, vec1: List[float], vec2: List[float]) -> float:
        """计算两向量的余弦距离（1 - 余弦相似度），取值 [0, 2]。"""
        if len(vec1) != len(vec2):
            raise ValueError("两个向量维度必须相同")

        mag1 = self._magnitude(vec1)
        mag2 = self._magnitude(vec2)

        if mag1 == 0 and mag2 == 0:
            return 0.0
        elif mag1 == 0 or mag2 == 0:
            return 1.0

        dot_prod = self._dot_product(vec1, vec2)
        cosine_similarity = dot_prod / (mag1 * mag2)
        cosine_similarity = max(-1.0, min(1.0, cosine_similarity))

        return 1.0 - cosine_similarity

    def cosineSimilarity(self, vec1: List[float], vec2: List[float]) -> float:
        """对外暴露的余弦距离，同 _cosine_distance。"""
        return 1.0-self._cosine_distance(vec1, vec2)

    def __len__(self) -> int:
        """返回索引中的文档数量。"""
        return len(self.vectors)

    def __repr__(self) -> str:
        """返回索引的简要描述（条数、维度、度量、是否含嵌入函数）。"""
        has_embed_fn = "是" if self._embedding_fn else "否"
        return f"VectorIndex(条数={len(self)}, 维度={self._vector_dim}, 度量='{self._distance_metric}', 含嵌入函数={has_embed_fn})"

In [369]:
with open("./RAG/report_cn.md", "r") as f:
    text = f.read()

In [370]:
# 1. 按章节对文本分块
chunks = chunk_by_section(text)
chunks = [c.strip() for c in chunks if c.strip()]
print(f"分块数量: {len(chunks)}")
for i, chunk in enumerate(chunks):
    print(f"chunk {i}:{chunk[:50]}")
    print("-"*50)

分块数量: 15
chunk 0:# **年度跨学科研究综述：跨领域洞察**
--------------------------------------------------
chunk 1:执行摘要

本报告综合了过去财年组织内各运营与研发部门的主要发现与持续研究进展。我们的优势在于思想与
--------------------------------------------------
chunk 2:目录

1. 执行摘要
2. 目录
3. 方法论
4. 第一节：医学研究——理解 XDR-471 综
--------------------------------------------------
chunk 3:方法论

本年度跨学科研究综述中的洞察来自标准部门报告周期、专项项目更新以及全年举行的跨职能评审会议
--------------------------------------------------
chunk 4:第一节：医学研究——理解 XDR-471 综合征

本年度我们在理解 XDR-471 综合征方面取得
--------------------------------------------------
chunk 5:第二节：软件工程——Project Phoenix 稳定性提升

软件工程部门投入大量精力提升支撑 
--------------------------------------------------
chunk 6:第三节：财务分析——第三季度业绩与展望

季度财务分析呈现复杂局面。集团整体收入同比小幅增长 3.1
--------------------------------------------------
chunk 7:第四节：科学实验——材料复合材料 XT-5 的表征

材料科学团队完成了材料复合材料 XT-5 的初
--------------------------------------------------
chunk 8:第五节：法律动态——应对知识产权判例与监管变化

法律部门本年度积极监测并应对多项重要进展。_Syn
--------------------------------------------------
chunk 9:第六节：产品工程——Model Zircon

In [371]:
# 2. 为每个分块生成嵌入向量（批量调用以避免限流）
embeddings = generate_embedding(chunks, input_type="document")
print(f"共生成 {len(embeddings)} 个嵌入向量，维度={len(embeddings[0])}")

共生成 15 个嵌入向量，维度=1024


In [372]:
# 3. 创建向量库并写入各嵌入（使用预计算的嵌入向量）
store = VectorIndex(embedding_fn=lambda q: generate_embedding(q, input_type="query"))
for i, chunk in enumerate(chunks):
    store.add_vector(embeddings[i], {"content": chunk,"index":i})
print(store)

VectorIndex(条数=15, 维度=1024, 度量='cosine', 含嵌入函数=是)


In [ ]:
# 4. 用户提出问题（search 接受字符串并在内部生成查询嵌入）
question = "第三季度财务表现如何？"


In [374]:
# 5. 在向量库中检索，返回最相关的 2 个分块
results = store.search(question, k=2)
for doc, distance in results:
    print(f"chunk {doc['index']}: [距离={distance:.4f}] {doc['content'][:200]}...")

chunk 6: [距离=0.5163] 第三节：财务分析——第三季度业绩与展望

季度财务分析呈现复杂局面。集团整体收入同比小幅增长 3.1%，主要由主要子公司在成熟市场的强劲表现驱动。然而，新兴市场部门出现轻微收缩（-1.5%），归因于竞争加剧和不利的汇率波动。多个关键产品线出现利润率侵蚀，与投入成本上升和供应链中断有关。旨在优化运营支出的 Project Hercules 实现了初步节约，但不足以完全抵消利润率压力。对研发项目的投资...
chunk 2: [距离=0.6444] 目录

1. 执行摘要
2. 目录
3. 方法论
4. 第一节：医学研究——理解 XDR-471 综合征
5. 第二节：软件工程——Project Phoenix 稳定性提升
6. 第三节：财务分析——第三季度业绩与展望
7. 第四节：科学实验——材料复合材料 XT-5 的表征
8. 第五节：法律动态——应对知识产权判例与监管变化
9. 第六节：产品工程——Model Zircon-5 规格定稿
...


In [375]:
# 测试代码
queryEmbedding = generate_embedding("第三季度财务表现如何？", input_type="query")
print(f"queryEmbedding length: {len(queryEmbedding)}")
idx =[4,5,6]
for i in idx:
    print(f"chunk{i} {chunks[i][:50]} 相似度: {store.cosineSimilarity(queryEmbedding, embeddings[i])}")
    print("-"*50)


queryEmbedding length: 1024
chunk4 第一节：医学研究——理解 XDR-471 综合征

本年度我们在理解 XDR-471 综合征方面取得 相似度: 0.27943563994046716
--------------------------------------------------
chunk5 第二节：软件工程——Project Phoenix 稳定性提升

软件工程部门投入大量精力提升支撑  相似度: 0.23470120386756954
--------------------------------------------------
chunk6 第三节：财务分析——第三季度业绩与展望

季度财务分析呈现复杂局面。集团整体收入同比小幅增长 3.1 相似度: 0.4836899172133833
--------------------------------------------------


In [379]:
results = store.search("what happened in INC-2023-Q4-011", k=2)
for doc, distance in results:
    print(f"chunk {doc['index']}: [距离={distance:.4f}] {doc['content'][:200]}...")

chunk 5: [距离=0.5811] 第二节：软件工程——Project Phoenix 稳定性提升

软件工程部门投入大量精力提升支撑 Project Phoenix 的核心系统的稳定性与性能。反复出现的问题，尤其是高峰负载下的 `ERR_MEM_ALLOC_FAIL_0x8007000E` 以及影响数据检索的 `TIMEOUT_QUERY_DB_0xDEADBEEF`，被列为优先处理项，对应事件成本为 INC-2023-Q4-01...
chunk 13: [距离=0.5859] 第十节：网络安全分析——事件响应报告：INC-2023-Q4-011

网络安全运营中心成功遏制并修复了编号为 `INC-2023-Q4-011` 的定向入侵尝试。威胁情报显示，该活动与 `ShadowNet Syndicate` 威胁行为者组织的战术、技术与程序相符。初始访问通过针对财务部门人员的鱼叉式钓鱼邮件获得，可能意在获取与第三节（财务分析）相关的数据。端点检测与响应（EDR）系统在工作站...
